In [0]:
# Apply Databricks system governed tags (class.*) to sensitive columns in formula1_incr.gold
# so ABAC column-mask / row-filter policies can target them with has_tag('class.<name>').
# Only base (MANAGED) tables are tagged; metric views read from these tables, so
# ABAC policies on the underlying columns are enforced through them.

CATALOG = "formula1_incr"
SCHEMA = "gold"
DRY_RUN = False  # set True to only print the statements

# table -> {column: system tag}
COLUMN_TAGS = {
    "dim_drivers": {
        "driver_name": "class.name",
        "date_of_birth": "class.date_of_birth",
        "nationality": "class.residency_status",  # citizenship of a person
        "nationality_region": "class.location",
    },
    "dim_constructors": {
        "nationality": "class.location",
        "nationality_region": "class.location",
    },
    "dim_races": {
        "locality": "class.location",
        "country": "class.location",
    },
    "ref_nationality_region": {
        "region": "class.location",
    },
}

# Validate tables/columns exist and are base tables before tagging
existing = {
    (r.table_name, r.column_name)
    for r in spark.sql(f"""
        SELECT c.table_name, c.column_name
        FROM {CATALOG}.information_schema.columns c
        JOIN {CATALOG}.information_schema.tables t
          ON c.table_schema = t.table_schema AND c.table_name = t.table_name
        WHERE c.table_schema = '{SCHEMA}' AND t.table_type IN ('MANAGED', 'EXTERNAL')
    """).collect()
}

results = []
for table, cols in COLUMN_TAGS.items():
    for col, tag in cols.items():
        if (table, col) not in existing:
            results.append((table, col, tag, "SKIPPED: column/table not found"))
            continue
        stmt = f"SET TAG ON COLUMN `{CATALOG}`.`{SCHEMA}`.`{table}`.`{col}` `{tag}`"
        if DRY_RUN:
            print(stmt)
            results.append((table, col, tag, "DRY_RUN"))
            continue
        try:
            spark.sql(stmt)
            results.append((table, col, tag, "APPLIED"))
        except Exception as e:
            results.append((table, col, tag, f"FAILED: {str(e)[:200]}"))

display(spark.createDataFrame(results, "table_name string, column_name string, tag string, status string"))

In [0]:
%sql
-- Confirm the class.* tags now present on gold columns
SELECT table_name, column_name, tag_name, tag_value
FROM formula1_incr.information_schema.column_tags
WHERE schema_name = 'gold'
  AND tag_name LIKE 'class.%'
ORDER BY table_name, column_name